In [77]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col
import scipy.stats as stats
from mpl_toolkits.mplot3d import Axes3D

#### Question 1
Explain the following terms (or concepts) in your own words.
##### a) A joint hypothesis: imposes two or more linear restrictions on the coefficients of the model
##### b) The F test: computes the heteroskedasicity-robust F-test; a hypothesis test that uses the F-distribution to evaluate multiple linear restrictions on regression coefficients simultaneously
##### c) A control variable: a regressor that accounts for an omitted factor affecting the dependent variable
##### d) The conditional mean independence assumption: once control variables are accounted for, the variables of interest become uncorrelated with the error term; this assumption ensures that the coefficients on the variables of interest have a causal interpretation
##### e) The dummy variable trap: occurs when the dummy variables corresponding to levels of an qualitative variable

#### Question 2
In this excerise, we will work with the data file birthweight_smoking.csv , which contains data for a random sample of babies born in Pennsylvania in 1989. The data includes the baby’s birth weight together with various characteristic of the mother, including whether she smoked during the pregnancy.
##### a) Run a regression of birthweight on smoker , alcohol , age , educ , and nprevist . Compute heteroskedasticity robust standard errors. Interpret the estimated coefficient on smoker.
##### b) Test for the joint significance of age and educ at the 5% significance level.
##### c) An alternative way to control for prenatal visits is to use the binary variables tripre1 through tripre3 . Regress birthweight on smoker , alcohol , age , educ , tripre0 , tripre1 , tripre2 and tripre3 . Interpret the estimated coefficient on tripre1 . Compare this to the estimated coefficient on nprevist in part (a).
##### d) Are the prenatal visit dummy variables jointly statistically significant at the 5% significance level?

In [78]:
BWsmoking = pd.read_csv('birthweight_smoking.csv')
BWsmoking.columns

Index(['nprevist', 'alcohol', 'tripre1', 'tripre2', 'tripre3', 'tripre0',
       'birthweight', 'smoker', 'unmarried', 'educ', 'age', 'drinks'],
      dtype='object')

In [79]:
model1 = smf.ols(formula = 'birthweight ~ smoker + alcohol + age + educ + nprevist', data = BWsmoking)
result1 = model1.fit(cov_type = 'HC1')
print(result1.summary())

                            OLS Regression Results                            
Dep. Variable:            birthweight   R-squared:                       0.074
Model:                            OLS   Adj. R-squared:                  0.072
Method:                 Least Squares   F-statistic:                     37.26
Date:                Sun, 10 May 2026   Prob (F-statistic):           3.58e-37
Time:                        22:21:20   Log-Likelihood:                -23292.
No. Observations:                3000   AIC:                         4.660e+04
Df Residuals:                    2994   BIC:                         4.663e+04
Df Model:                           5                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   2924.9630     80.086     36.523      0.0

##### a) The estimated coefficient on smoker is -206.5068, meaning that if the mother did smoke during the pregnancy, the birth weight would decrease by about 206.5068 grams, holding everything else constant. 

In [80]:
hypothesis = "age = 0, educ = 0"
f_test_result1 = result1.f_test(hypothesis)
print(f_test_result1)

<F test: F=1.8096936487922681, p=0.1638833019126929, df_denom=2.99e+03, df_num=2>


##### b) Since F=1.81 < 2.996 and p=0.16 > 0.05, we fail to reject the null hypothesis at the 5% significance level; age and educ are not jointly statisically significant. 

In [81]:
model2 = smf.ols(formula = 'birthweight ~ smoker + alcohol + age + educ + tripre0 + tripre1 + tripre2 + tripre3', data=BWsmoking)
result2 = model2.fit(cov_type = 'HC1')
print(result2.summary())

                            OLS Regression Results                            
Dep. Variable:            birthweight   R-squared:                       0.049
Model:                            OLS   Adj. R-squared:                  0.047
Method:                 Least Squares   F-statistic:                 1.322e+04
Date:                Sun, 10 May 2026   Prob (F-statistic):               0.00
Time:                        22:21:53   Log-Likelihood:                -23332.
No. Observations:                3000   AIC:                         4.668e+04
Df Residuals:                    2992   BIC:                         4.673e+04
Df Model:                           7                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   2410.8940     66.213     36.411      0.0

##### c) The estimated coefficient on tripre1 is about 819.89, meaning that relative to mothers who don't go to any prenatal visits, mothers who go to the prenatal visit in the first trimester, their babies weigh about 819.89 grams more, holding everything else constant. In comparison, the coefficient for nprevisit in model1 is about 33, meaning that for each additional vist, the birth weight would increase by about 33 grams. However, comparing the coefficient of nprevisit in model1 and tripre1 in model2 is somewhat unfair because you can't compare them directly since nprevisit is the amount of times you went in for a prenatal visit while tripre1 is a dummy for if you went for a prenatal visit in your first trimester. So instead, comparing the differences between tripre1 and tripre2 as well as tripre2 and tripre3 and taking the average of the two would be a more comparable number to nprevisit. Also, it is more effective to only use the nprevisit variable instead of all the dummies because its value of R-squared is higher, 0.074 > 0.049.

In [82]:
hypothesis = "tripre0 = 0, tripre1 = 0, tripre2 = 0, tripre3 = 0"
f_test_result2 = result2.f_test(hypothesis)
print(f_test_result2)

<F test: F=430.76699034417027, p=1.7341778708158674e-293, df_denom=2.99e+03, df_num=4>


In [83]:
cv_d = stats.chi2.ppf(0.95, df=4) / 4
print(cv_d)

2.3719322591952885


##### d) The prenatal visit dummy variables are jointly statistically significant at the 5% level because F=430.77 > 2.372 and p ≈ 0 < 0.05 . 

#### Question 3
Each month the Bureau of Labor Statistics in the U.S. Department of Labor conducts the
Current Population Survey (CPS), which provides data on labor force characteristics of the population,
including the level of employment, unemployment, and earnings. Approximately 65,000 randomly
selected U.S. households are surveyed each month. The sample is chosen by randomly selecting
addresses from a database comprised of addresses from the most recent decennial census augmented
with data on new housing units constructed after the last census. The exact random sampling scheme
is rather complicated (first small geographical areas are randomly selected, then housing units within
these areas randomly selected); details can be found in the Handbook of Labor Statistics and is
described on the Bureau of Labor Statistics website (www.bls.gov). For this exercise, you
will utilize the
data set called cpsDATA.csv .
##### a) Run a regression of log (average hourly) earnings log(ahe) on education yrseduc , female and age . Compute heteroskedasticity robust standard errors. What is the estimated coefficient on yrseduc ?
##### b) Interpret the estimated coefficient on age . If age increases from 25 to 26, how are earnings expected to change? If age increases from 33 to 34, how are earnings expected to change?
##### c) Run a regression of log(ahe) on education yrseduc , female , age and $age^2$ . Compute heteroskedasticity robust standard errors.
##### d) What is the marginal effect of age on earnings if age increases from 25 to 26, how are earnings expected to change? If age increases from 33 to 34, how are earnings expected to change?

In [84]:
CPSData = pd.read_csv('cpsDATA.csv')
CPSData["log_ahe"] = np.log(CPSData["ahe"])
CPSData["age2"] = CPSData["age"] ** 2
CPSData.head()

,ahe,yrseduc,female,age,northeast,midwest,south,west,log_ahe,age2
0,21.367521,13,1,65,1,0,0,0,3.061872,4225
1,24.038462,16,0,56,1,0,0,0,3.179655,3136
2,7.692307,12,0,42,1,0,0,0,2.040221,1764
3,15.182186,12,1,42,1,0,0,0,2.720123,1764
4,20.614346,14,1,53,1,0,0,0,3.025987,2809


In [85]:
model1 = smf.ols(formula = 'log_ahe ~ yrseduc + female + age', data = CPSData)
result1 = model1.fit(cov_type = 'HC1')
print(result1.summary())

                            OLS Regression Results                            
Dep. Variable:                log_ahe   R-squared:                       0.279
Model:                            OLS   Adj. R-squared:                  0.279
Method:                 Least Squares   F-statistic:                     7307.
Date:                Sun, 10 May 2026   Prob (F-statistic):               0.00
Time:                        22:23:27   Log-Likelihood:                -47114.
No. Observations:               59668   AIC:                         9.424e+04
Df Residuals:                   59664   BIC:                         9.427e+04
Df Model:                           3                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      1.1566      0.014     81.078      0.0

In [86]:
application_table = summary_col(results = [result1],
                               float_format = '%0.3f',
                               stars = True,
                               model_names = ['Model (1)'])
application_table

,Model (1)
Intercept,1.157***
,(0.014)
yrseduc,0.112***
,(0.001)
female,-0.238***
,(0.004)
age,0.009***
,(0.000)
R-squared,0.279
R-squared Adj.,0.279


##### a) The estimated coefficient of on yrseduc is 0.1121, meaning that for every 1 unit increase in years of education while holding age and gender constant, there is an 11.21% increase in average hourly earnings. 
##### b) The coefficient on age is 0.00929, meaning for every additional year, average hourly earning increased by 0.929%. This means that when age increases from 25 to 26 and from 33 to 34, both times earnings will increase by 0.929%. (linear function)

In [87]:
application_table = summary_col(results = [result2],
                               float_format = '%0.3f',
                               stars = True,
                               model_names = ['Model ()'])
application_table

,Model ()
Intercept,2410.894***
,(66.213)
smoker,-211.895***
,(27.825)
alcohol,-28.949
,(70.441)
age,2.798
,(2.374)
educ,10.926*
,(5.601)


##### c) As shown above with the regression summary and the chart analysis.

In [54]:
b1 = result2.params['age']
b2 = result2.params['age2']
print(f"β̂₃ (age):  {b1:.6f}")
print(f"β̂₄ (age²): {b2:.6f}")

for a0, a1 in [(25, 26), (33, 34)]:
    delta = b1 * (a1 - a0) + b2 * (a1**2 - a0**2)
    print(f"Age {a0}→{a1}: Δlog(ahe) = {delta:.6f}  →  ≈ {delta*100:.4f}% change")

β̂₃ (age):  0.051037
β̂₄ (age²): -0.000472
Age 25→26: Δlog(ahe) = 0.026971  →  ≈ 2.6971% change
Age 33→34: Δlog(ahe) = 0.019421  →  ≈ 1.9421% change


##### d) The expected earnings change from age 25 to 26 is abotu 2.70% while from age 33 to 34, it's expected change is 1.94%. Here the expected earnings change is not the same as the previous model because this function is not linear.

#### Question 4
In this exercise, you will work with the data file lead_mortality.csv . This file contains a subset of the data set used in Clay, Troesken, and Haines (2014). Lead is toxic, particularly for young children, and for this reason, government regulations severely restrict the amount of lead in the environment. In the early part of the 20th century, the underground water pipes in many U.S. cities contained lead, and lead from these pipes leached into drinking water. You will investigate the effect of these lead water pipes on infant mortality.
##### a) Create a table of descriptive statistics for the numeric variables in the data set. What is the average infant mortality rate in the sample? What is the average pH level in the sample?
##### b) The amount of lead leached from lead pipes depends on the chemistry of the water running through the pipes. The more acidic the water is (that is, the lower its pH), the more lead is leached. To this end, consider the following regression model.
infrate = β0 + β1lead + β2ph + β3 (lead × ph) + u.
Explain what each coefficient measures and the role of the interaction term (assume that the zero
conditional mean assumption holds).
##### c) Estimate the model in part (b) and compute heteroskedasticity robust standard errors.
##### d) What is the estimated effect of lead on infant mortality at the sample mean of ph ?
##### e) Suppose the ph level is one standard deviation lower than the average level of ph in the sample. What is the estimated effect of lead on infant mortality?

In [88]:
LMdata = pd.read_csv('lead_mortality.csv')
LMdata.head()

,year,city,state,age,hardness,ph,infrate,typhoid_rate,np_tub_rate,mom_rate,population,precipitation,temperature,lead,foreign_share
0,1900,Alameda,CA,28.954838,97,7.6,0.109756,0.024390,0.030488,0.195122,164.0,1.850307,59.026169,0,0.193548
1,1900,Albany,NY,30.347675,43,7.3,0.298618,0.041445,0.013815,0.179596,941.0,3.278735,45.267910,1,0.184884
2,1900,Allegheny,PA,27.087301,111,7.3,0.446841,0.093991,0.027735,0.192604,1298.0,3.350943,48.971493,1,0.237302
3,1900,Allentown,PA,27.764046,176,7.7,0.384181,0.028249,0.005650,0.197740,354.0,3.350943,48.971493,1,0.078652
4,1900,Altoona,PA,27.037533,111,7.3,0.467866,0.043702,0.007712,0.182519,389.0,3.350943,48.971493,0,0.112601


In [89]:
num_cols = LMdata.select_dtypes(include=['number']).columns.tolist()
desc_stats = LMdata[num_cols].describe().T[
    ['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']
]
desc_stats.round(4)

,count,mean,std,min,25%,50%,75%,max
year,172.0,1900.0000,0.0000,1900.0000,1900.0000,1900.0000,1900.0000,1900.0000
age,172.0,27.7047,2.2169,22.0125,26.0551,27.5892,28.9943,34.4439
hardness,172.0,113.4360,98.7586,2.0000,30.0000,96.5000,148.0000,445.0000
ph,172.0,7.3227,0.6917,5.7000,6.7000,7.5000,7.9250,8.9000
infrate,172.0,0.3962,0.1513,0.1098,0.2934,0.3864,0.4703,0.8445
typhoid_rate,172.0,0.0411,0.0285,0.0000,0.0229,0.0338,0.0484,0.1443
np_tub_rate,172.0,0.0205,0.0120,0.0000,0.0121,0.0190,0.0278,0.0574
mom_rate,172.0,0.1990,0.0321,0.1309,0.1824,0.1967,0.2157,0.3237
population,172.0,1087.9624,3137.3147,85.0000,215.7500,357.5000,806.5000,34355.5391
precipitation,172.0,3.3286,0.5792,0.9546,3.1935,3.3658,3.6138,4.7541


##### a) The average infant mortality rate in the sample is 0.3962, meaning that approximately 396 infant deaths per 1,000 live births. The average pH in the sample is 7.3227.

##### b) β₀, being the intercept, would be the amount of infant deaths if there were no lead pipes and a pH of 0. β₁ would be how many more or less infant deaths there would be if there are lead pipes while ph=0. β₂ would be how many more or less infant deaths there would be depending on the pH level while lead=0. β₃ would be the effect of lead pipes and pH levels on infant mortality given both are not 0.

In [90]:
model1 = smf.ols(formula = 'infrate ~ lead + ph + (lead*ph)', data = LMdata )
result1 = model1.fit(cov_type = 'HC1')
print(result1.summary())

                            OLS Regression Results                            
Dep. Variable:                infrate   R-squared:                       0.272
Model:                            OLS   Adj. R-squared:                  0.259
Method:                 Least Squares   F-statistic:                     20.97
Date:                Sun, 10 May 2026   Prob (F-statistic):           1.37e-11
Time:                        22:29:13   Log-Likelihood:                 108.52
No. Observations:                 172   AIC:                            -209.0
Df Residuals:                     168   BIC:                            -196.5
Df Model:                           3                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.9189      0.150      6.106      0.0

In [91]:
application_table = summary_col(results = [result1],
                               float_format = '%0.4f',
                               stars = True,
                               model_names = ['Model (1)'])
application_table

,Model (1)
Intercept,0.9189***
,(0.1505)
lead,0.4618**
,(0.2076)
ph,-0.0752***
,(0.0210)
lead:ph,-0.0569**
,(0.0281)
R-squared,0.2719
R-squared Adj.,0.2589


##### c) The estimated model is 'infrate = 0.9189 + 0.4618lead - 0.0752ph - 0.569(lead*ph)'. 

In [92]:
b1 = result1.params['lead']
b3 = result1.params['lead:ph']

ph_mean = LMdata['ph'].mean()

effect_mean_ph = b1 + b3 * ph_mean
print(f"β̂₁ (lead):    {b1:.6f}")
print(f"β̂₃ (lead:ph): {b3:.6f}")
print(f"Sample mean of ph: {ph_mean:.4f}")
print(f"Effect of lead at mean ph = {b1:.4f} + ({b3:.4f}) × {ph_mean:.4f}")
print(f"                         = {effect_mean_ph:.6f}")

β̂₁ (lead):    0.461798
β̂₃ (lead:ph): -0.056862
Sample mean of ph: 7.3227
Effect of lead at mean ph = 0.4618 + (-0.0569) × 7.3227
                         = 0.045415


##### d) The estimated effect of lead on infant mortality at the sample mean of ph is approximately +0.0454. 

In [93]:
ph_std = LMdata['ph'].std()

ph_low = ph_mean - ph_std

effect_low_ph = b1 + b3 * ph_low

print(f"Sample mean of ph:         {ph_mean:.4f}")
print(f"Std deviation of ph:       {ph_std:.4f}")
print(f"ph (mean - 1 SD):          {ph_low:.4f}")
print()
print(f"Effect at mean ph:        {effect_mean_ph:.6f}")
print(f"Effect at (mean - 1 SD):  {effect_low_ph:.6f}")
print(f"Difference (amplification): {effect_low_ph - effect_mean_ph:.6f}")


Sample mean of ph:         7.3227
Std deviation of ph:       0.6917
ph (mean - 1 SD):          6.6309

Effect at mean ph:        0.045415
Effect at (mean - 1 SD):  0.084748
Difference (amplification): 0.039333


##### e) If the ph is one standard deviation below, at 6.6309, then that changes the outcome to approximately +0.0393.

Some AI was used to fix code as it wasn't running as it kept getting errors that I couldn't fix.